In [1]:
!nvidia-smi
!python -V

In [2]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/likhithashree01-beep/Habitat.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/Habitat")

os.environ["HABITAT_REPO_URL"] = REPO_URL
os.environ["HABITAT_REPO_BRANCH"] = REPO_BRANCH
os.environ["HABITAT_PROJECT_ROOT"] = str(PROJECT_ROOT)

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH:", REPO_BRANCH)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
%%bash
set -e

REPO_URL="$HABITAT_REPO_URL"
REPO_BRANCH="$HABITAT_REPO_BRANCH"
PROJECT_ROOT="$HABITAT_PROJECT_ROOT"

if [[ -z "$REPO_URL" ]]; then
  echo "Set REPO_URL before running this cell."
  exit 1
fi

if [ ! -d "$PROJECT_ROOT/.git" ]; then
  rm -rf "$PROJECT_ROOT"
  git clone --branch "$REPO_BRANCH" "$REPO_URL" "$PROJECT_ROOT"
else
  cd "$PROJECT_ROOT"
  git fetch origin "$REPO_BRANCH"
  git checkout "$REPO_BRANCH"
  git pull --ff-only origin "$REPO_BRANCH"
fi

cd "$PROJECT_ROOT"
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD

In [4]:
expected = [
    "code",
    "configs",
    "datasets",
    "notebooks",
    "outputs",
]

missing = [name for name in expected if not (PROJECT_ROOT / name).exists()]
if missing:
    print("Missing repo folders:", missing)
    print("Creating them so environment setup can continue.")

for rel in [
    "code/agents",
    "code/runners",
    "code/utils",
    "configs",
    "datasets",
    "outputs/logs",
    "outputs/videos",
    "notebooks",
]:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())
print("Top-level folders:", sorted(p.name for p in PROJECT_ROOT.iterdir()))

In [5]:
%%bash
set -e

if [ ! -d "/opt/conda" ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
  bash Miniconda3-latest-Linux-x86_64.sh -b -p /opt/conda
fi

source /opt/conda/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
conda --version

In [6]:
import os
os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]
print(os.environ["PATH"].split(":")[:4])

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh

if ! conda env list | awk '{print $1}' | grep -qx habitatEnv; then
  conda create -y -n habitatEnv -c conda-forge python=3.9
fi

conda activate habitatEnv

pip install -U pip
pip install "pillow==10.4.0"
conda install -y -c conda-forge -c aihabitat habitat-sim withbullet headless
pip install "git+https://github.com/facebookresearch/habitat-lab.git#subdirectory=habitat-lab"


python -c "import habitat_sim; print('habitat_sim OK', habitat_sim.__version__)"
python -c "import habitat; print('habitat OK', habitat.__version__)"
python -c "import PIL; print('Pillow', PIL.__version__)"

## Configure Headless EGL

Colab needs an explicit EGL vendor file so Habitat-Sim can render without a display.


In [8]:
%%bash
set -e

NVIDIA_EGL_LIB=$(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -n 1)
echo "Using: $NVIDIA_EGL_LIB"

cat > /tmp/nvidia_egl.json <<EOF
{
  "file_format_version": "1.0.0",
  "ICD": { "library_path": "$NVIDIA_EGL_LIB" }
}
EOF

echo "EGL ICD created at /tmp/nvidia_egl.json"

In [9]:
import glob
import os

lib = glob.glob("/usr/**/libEGL_nvidia.so.0", recursive=True)[0]
os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = "/tmp/nvidia_egl.json"
os.environ["LD_LIBRARY_PATH"] = os.path.dirname(lib) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["NVIDIA_DRIVER_CAPABILITIES"] = "all"

print("EGL forced to:", lib)
print("EGL JSON:", os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"])

In [10]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
import habitat
import habitat_sim

print('Habitat-Lab import success')
print('Habitat-Sim import success')
print('habitat version:', habitat.__version__)
print('habitat_sim version:', habitat_sim.__version__)
PY

In [11]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

SCENE_DIR=/content/Habitat/datasets/test_scenes
mkdir -p "$SCENE_DIR"
cd "$SCENE_DIR"

if [ ! -f "skokloster-castle.glb" ] || [ ! -s "skokloster-castle.glb" ]; then
  echo "Downloading habitat-test-scenes.zip..."
  rm -f skokloster-castle.glb skokloster-castle.navmesh habitat-test-scenes.zip
  wget -q http://dl.fbaipublicfiles.com/habitat/habitat-test-scenes.zip
  unzip -q -o habitat-test-scenes.zip
  find . -name "skokloster-castle.glb" -exec mv {} . \;
  find . -name "skokloster-castle.navmesh" -exec mv {} . \;
  rm -f habitat-test-scenes.zip
fi

ls -lh skokloster-castle.glb skokloster-castle.navmesh

In [12]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
import os
import habitat_sim
from habitat_sim import Configuration, SimulatorConfiguration
from habitat_sim.agent import AgentConfiguration
from habitat_sim.sensor import CameraSensorSpec, SensorType

scene_path = "/content/Habitat/datasets/test_scenes/skokloster-castle.glb"
assert os.path.exists(scene_path), f"Missing scene: {scene_path}"

sim_cfg = SimulatorConfiguration()
sim_cfg.scene_id = scene_path
sim_cfg.enable_physics = False

rgb_spec = CameraSensorSpec()
rgb_spec.uuid = "rgb"
rgb_spec.sensor_type = SensorType.COLOR
rgb_spec.resolution = [256, 256]
rgb_spec.position = [0.0, 1.5, 0.0]
rgb_spec.hfov = 90.0

agent_cfg = AgentConfiguration()
agent_cfg.sensor_specifications = [rgb_spec]

cfg = Configuration(sim_cfg, [agent_cfg])
sim = habitat_sim.Simulator(cfg)
obs = sim.get_sensor_observations()
rgb = obs["rgb"]

print("Obs keys:", list(obs.keys()))
print("RGB shape:", rgb.shape)
print("RGB dtype:", rgb.dtype)
print("RGB min/max:", rgb.min(), rgb.max())

sim.close()
print("Direct-config headless render OK")
PY

In [13]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
from habitat.config.default import get_config

cfg = get_config("benchmark/nav/pointnav/pointnav_hm3d.yaml")

print("=== DATASET CONFIG ===")
print("type:", cfg.habitat.dataset.type)
print("split:", cfg.habitat.dataset.split)
print("scenes_dir:", cfg.habitat.dataset.scenes_dir)
print("data_path:", cfg.habitat.dataset.data_path)

print("\n=== SENSOR SETUP ===")
print(list(cfg.habitat.simulator.agents.main_agent.sim_sensors.keys()))

print("\nPointNav config load OK")
PY

In [14]:
from pathlib import Path

dataset_dirs = [
    PROJECT_ROOT / "datasets" / "scene_datasets",
    PROJECT_ROOT / "datasets" / "pointnav",
    PROJECT_ROOT / "datasets" / "hm3d",
]

for path in dataset_dirs:
    path.mkdir(parents=True, exist_ok=True)
    print(path)

print("\nDataset directories are ready. Download cells can target these paths in the next notebook.")